In [9]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,  pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from evaluate import evaluator

task_evaluator = evaluator("text-generation")

summary_prompt = """You are an expert radiologist. You are about to see the 'findings' section of a {modality} report. Please create an appropriate summary/conclusion for these findings. 
Begin your final response with "<Impression> and end it with </Impression>".
Report findings: {report}
"""

modality = "CT chest"

In [10]:
df = pd.read_csv("data/mimic_rrs/mimic-CT_chest-val.csv", index_col=0)
example = df['finding'][0]
df

,finding,impression
0,no filling defect is identified within the pul...,1. no evidence of pulmonary embolus. 2. multif...
1,the technical quality of the examination is go...,no evidence of acute or chronic pe. no aortic ...
2,there is no evidence for pulmonary embolus. th...,1. no evidence for pulmonary embolus. moderate...
3,ct of the chest with contrast: a nodule measur...,1. left pleural effusion stable. 2. new left p...
4,there is no filling defect in pulmonary artery...,1. negative examination for pulmonary embolism...
...,...,...
1273,scattered axillary and mediastinal lymph nodes...,1. diffuse bilateral peribronchovascular groun...
1274,there is no evidence of pulmonary embolism to ...,1. no evidence of pulmonary embolism. 2. conti...
1275,there is no pulmonary embolism. there is no pn...,1. unchanged appearance of left lower lobe col...
1276,the heart and pericardium are unremarkable wit...,1. no dissection or pulmonary embolism is iden...


In [12]:
model_id = "google/medgemma-1.5-4b-it"
# model_id = "openai/gpt-oss-20b"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     torch_dtype="auto",
#     device_map="auto",
#     quantization_config=quantization_config
# )

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=pipe)

llm_= ChatHuggingFace(llm=llm)

output = llm.invoke(f"Summarize the following report: {summary_prompt.format(modality=modality, report=example)}")


print(output)




Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize the following report: You are an expert radiologist. You are about to see the 'findings' section of a CT chest report. Please create an appropriate summary/conclusion for these findings. 
Begin your final response with "<Impression> and end it with </Impression>".
Report findings: no filling defect is identified within the pulmonary arteries. the thoracic aorta maintains a normal contour. the heart and pericardium are within normal limits. lungs show moderate bilateral pleural effusions, slightly increased since prior exam. multifocal airspace opacities are seen throughout both lungs consistent with pneumonia, slightly better aerated in the upper lobes compared to ___. since the last exam there is interval development of bronchietasis within the lingula, left upper lobe and right middle lobe. small mediastinal lymph nodes are present, the largest of which measures 1.2 cm in the left paratracheal station. limited views of the upper abdomen show ascites.
<Impression>
The patien

In [5]:
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
print("pipeline module:", pipeline.__module__)

transformers version: 5.3.0
transformers file: /home/stephen/langchain/lib/python3.11/site-packages/transformers/__init__.py
pipeline module: transformers.pipelines
